# Chapter 8 lab — How much room remains after pending approvals?

Draft educator companion v1 · 8 September 2026 · 90 minutes.

Read [the chapter](https://www.profrod.ai/book/ch08-approval) alongside this lab. **Prerequisites:** Chapter 7 work records; exact proposals and reserved versus spent money.

You will build one explicitly scoped decision function, challenge it with independently authored cases, and trace the same concern through the cumulative runtime. The manuscript is where you build the full components; this notebook is a focused companion, not a claim that importing a runtime teaches its construction.

**Before running:** write your prediction. Keep the worked solution below closed until you have attempted the function. Download/open this notebook in an existing Jupyter environment using the book’s Python 3.14 interpreter after completing repository setup in the book conventions. Unlike the two Chapter 1 notebooks, this lab requires the local checkout and its locked book dependencies. It does not install packages, launch a hosted notebook, or require model credentials.

**Without a notebook server:** read and edit the cells in your editor, then run `uv run --python 3.14 python book/always_on/educator/run_lesson_v1.py --chapter 8 --output /tmp/lucy-ch08-class.json` from the repository root. Use a new output filename on each retained run. The runner executes the saved notebook and records student results separately from the worked example.


## 1. Predict (10 minutes)

Lucy has a 2500-pence limit. Vanilla reserves 1750 and strawberry would reserve 1100. Both individual orders are smaller than the limit. Can both be approved?

Write both the expected result and the evidence that could disprove your explanation.


In [ ]:
import copy
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("Use the book Python 3.14 environment for Chapters 2–16.")
# Open the notebook inside your source checkout, or set this path explicitly.
start = Path(os.environ.get("SOVEREIGN_AGENT_REPO", Path.cwd())).resolve()
ROOT = next(
    (p for p in (start, *start.parents) if (p / "book/always_on/checkpoints/ch08.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Set SOVEREIGN_AGENT_REPO to the Sovereign Agent checkout.")
CHECKPOINT = ROOT / "book/always_on/checkpoints/ch08.py"
EXPECTED_CHECKPOINT_SHA256 = "09967c440089ed7378bb4b77ae6b5788a614c8387091db6a3079d5058b4ec95d"
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError("Checkpoint version differs from this lesson; use its matching release.")
print("Chapter 8 checkpoint bytes match this lesson. No model or channel has been called.")

## 2. Build your decision (25 minutes)

Implement decide(case) as a reservation admission calculation over trusted integer pence. Return REFUSED if amount <= 0 or spent+reserved+amount > limit; otherwise return the new reserved amount. An existing grant must be deduplicated by operation ID before calling this function; this function deliberately does not implement idempotency.

`decide` is your code. `grade` and the fixtures are supplied test infrastructure. The examples below specify expected answers independently; do not generate those answers with your function. An unimplemented starter is reported as NOT_SUBMITTED, never as a pass.


In [ ]:
def grade(candidate, cases):
    results = []
    for index, (case, expected) in enumerate(cases, 1):
        supplied = copy.deepcopy(case)
        try:
            observed = candidate(supplied)
        except NotImplementedError:
            results.append({"case": index, "status": "NOT_SUBMITTED"})
            continue
        except Exception as error:
            results.append({"case": index, "status": "FAILED", "error_type": type(error).__name__})
            continue
        try:
            passed = json.dumps(observed, sort_keys=True, allow_nan=False) == json.dumps(
                expected, sort_keys=True, allow_nan=False
            ) and json.dumps(supplied, sort_keys=True, allow_nan=False) == json.dumps(
                case, sort_keys=True, allow_nan=False
            )
        except (TypeError, ValueError):
            passed = False
        results.append(
            {
                "case": index,
                "status": "PASS" if passed else "FAILED",
                "expected": expected,
                "observed": observed,
            }
        )
    return results


def decide(case):
    # Replace this body with your implementation of the contract above.
    raise NotImplementedError("Write your function before consulting the worked solution.")

In [ ]:
CASES = [
    ({"spent": 0, "reserved": 0, "amount": 1750, "limit": 2500}, 1750),
    ({"spent": 0, "reserved": 1750, "amount": 1100, "limit": 2500}, "REFUSED"),
    ({"spent": 1500, "reserved": 0, "amount": 1000, "limit": 2500}, 1000),
    ({"spent": 1500, "reserved": 0, "amount": 1001, "limit": 2500}, "REFUSED"),
    ({"spent": 0, "reserved": 0, "amount": 0, "limit": 2500}, "REFUSED"),
]
submission_results = grade(decide, CASES)
print(json.dumps(submission_results, indent=2))

## 3. Inspect and run the cumulative reference (20 minutes)

checkpoints/ch08.py: experiment. Follow digest, approve, proposal revision, execute and independent supplier rows.

Open the named code before running it. Point to where an input reaches a decision and where that decision changes an observable result. The next cell executes the supplied chapter checkpoint; it is reference evidence, not a substitute for your implementation. Local supplier and worker processes use temporary state and are cleaned up by the checkpoint. Chapter 11 also launches a bounded local MCP process. Chapter 15 does not install a system service.


In [ ]:
# This supplied cumulative program is separate from grading your function.
# It uses fixture models/channels. Some chapters start local child processes.
# No --live, --telegram or --containers switch is added.
reference_environment = {
    k: v for k, v in os.environ.items() if k in {"PATH", "SYSTEMROOT", "TMPDIR", "LANG", "LC_ALL"}
}
reference_environment["PYTHONPATH"] = str(ROOT / "src")
reference_run = subprocess.run(
    [sys.executable, str(CHECKPOINT)],
    cwd=ROOT,
    env=reference_environment,
    capture_output=True,
    text=True,
    timeout=180,
    check=True,
)
print(reference_run.stdout)
EXPECTED_OBSERVATIONS = [
    "Cumulative overspend refused: True",
    "Supplier orders after authorized send: 1",
]
assert all(text in reference_run.stdout for text in EXPECTED_OBSERVATIONS)
print("REFERENCE_CHECKPOINT_PASSED — this is not your submission grade.")

## 4. Transfer the rule (20 minutes)

Run the real approval experiment and find the two identical approve calls. Why must the reservation remain 1750 rather than 3500? Then change an order’s quantity and identify which digest Lucy must approve.

Add your new case to TRANSFER_CASES, with an independently calculated expected answer. A blank list means the transfer remains unsubmitted. Describe one limit of your function before comparing it with the runtime.


In [ ]:
TRANSFER_CASES = []  # Add (input, expected) pairs after writing your prediction.
transfer_results = grade(decide, TRANSFER_CASES)
print(json.dumps(transfer_results, indent=2) if transfer_results else "TRANSFER_NOT_SUBMITTED")

## 5. Worked solution — reveal after attempting the task

The following function is an answer key, not a replacement for your submission. Its case results are recorded separately. Your teacher grades your original function, explanation and transfer case.

Approval binds the exact operation/proposal digest. Repeating that grant does not reserve again. Revised vanilla quantity seven invalidates the old six-tub proposal and releases its obsolete reservation. The supplier ends with one order for seven tubs and spending (reserved=0, spent=1750).


In [ ]:
def worked_decide(case):
    if case["amount"] <= 0 or case["spent"] + case["reserved"] + case["amount"] > case["limit"]:
        return "REFUSED"
    return case["reserved"] + case["amount"]


worked_results = grade(worked_decide, CASES)
assert all(row["status"] == "PASS" for row in worked_results)
print("WORKED_EXAMPLE_PASSED; submission_results remains separate.")

## 6. Break the tempting implementation (10 minutes)

Explain why the following shortcut violates at least one case. Predict which case catches it before running. Then name a different defect the current cases might miss.


In [ ]:
def tempting_shortcut(case):
    return case["reserved"] + case["amount"] if case["amount"] <= case["limit"] else "REFUSED"


shortcut_results = grade(tempting_shortcut, CASES)
assert any(row["status"] == "FAILED" for row in shortcut_results)
print(json.dumps(shortcut_results, indent=2))

## Exit ticket (5 minutes)

Submit your prediction, original decide function, case results, one transfer case, and the runtime path you traced. Explain: (1) which boundary Python enforced, (2) what evidence came from the supplied program, and (3) what remains unproved.

**Misconception to resolve:** A per-order ceiling alone permits cumulative overspending. Consent in a conversation is not an execution-time grant for arbitrary revised content.

**Scope of this lab:** The exercise is arithmetic only. Authentication, expiry, revocation, exact bytes and transaction atomicity remain required in the real approval path.

A successful reference run or worked example does not establish learner mastery. Instructor guidance and answers are in the matching versioned guide.
